Bringing in pandas, the only library needed for this cleanup.

In [1]:
import pandas as pd

Loading the three raw CSVs — orders, order details, and the sales target sheet — straight from the data/raw folder.

In [2]:
orders = pd.read_csv("data/raw/list_of_orders.csv")
order_details = pd.read_csv("data/raw/order_details.csv")
sales_target = pd.read_csv("data/raw/sales_target.csv")

### data cleaning


Checking column names, data types, and row count for the orders table before touching anything.

In [3]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 560 entries, 0 to 559
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   Order ID      500 non-null    str  
 1   Order Date    500 non-null    str  
 2   CustomerName  500 non-null    str  
 3   State         500 non-null    str  
 4   City          500 non-null    str  
dtypes: str(5)
memory usage: 22.0 KB


Peeking at the first few rows to see what the data actually looks like.

In [4]:
orders.head()

,Order ID,Order Date,CustomerName,State,City
0,B-25601,01-04-2018,Bharat,Gujarat,Ahmedabad
1,B-25602,01-04-2018,Pearl,Maharashtra,Pune
2,B-25603,03-04-2018,Jahan,Madhya Pradesh,Bhopal
3,B-25604,03-04-2018,Divsha,Rajasthan,Jaipur
4,B-25605,05-04-2018,Kasheen,West Bengal,Kolkata


Order Date comes in as text, so this converts it into a real datetime column.

In [5]:
orders['Order Date'] = pd.to_datetime(orders['Order Date'], format = '%d-%m-%Y') 

Counting nulls column by column to see where the gaps are.

In [6]:
orders.isnull().sum()

Order ID        60
Order Date      60
CustomerName    60
State           60
City            60
dtype: int64

Dropping any row that's completely empty across every column.

In [7]:
orders = orders.dropna(how= "all")

Order ID is the key we merge on later, so rows missing it get dropped here.

In [8]:
orders = orders.dropna(subset= ["Order ID"])

Quick check for exact duplicate rows.

In [9]:
orders.duplicated().sum()

np.int64(0)

State and City have inconsistent spacing and casing, so this trims whitespace and title-cases them.

In [10]:
for col in ["State", "City"]:
    orders[col] = orders[col].astype(str).str.strip().str.title()

Same info check as before, this time for the order_details table.

In [11]:
order_details.info()

<class 'pandas.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Order ID      1500 non-null   str    
 1   Amount        1500 non-null   float64
 2   Profit        1500 non-null   float64
 3   Quantity      1500 non-null   int64  
 4   Category      1500 non-null   str    
 5   Sub-Category  1500 non-null   str    
dtypes: float64(2), int64(1), str(3)
memory usage: 70.4 KB


Null count for order_details.

In [12]:
order_details.isnull().sum()

Order ID        0
Amount          0
Profit          0
Quantity        0
Category        0
Sub-Category    0
dtype: int64

Duplicate check for order_details.

In [13]:
order_details.duplicated().sum()

np.int64(0)

Cleaning up Category and Sub-Category text the same way — strip and title case.

In [14]:
for col in ["Category", "Sub-Category"]:
    order_details[col] = order_details[col].astype(str).str.strip().str.title()

Preview of order_details after cleaning.

In [15]:
order_details.head()

,Order ID,Amount,Profit,Quantity,Category,Sub-Category
0,B-25601,1275.0,-1148.0,7,Furniture,Bookcases
1,B-25601,66.0,-12.0,5,Clothing,Stole
2,B-25601,8.0,-2.0,3,Clothing,Hankerchief
3,B-25601,80.0,-56.0,4,Electronics,Electronic Games
4,B-25602,168.0,-111.0,2,Electronics,Phones


Merging orders and order_details on Order ID into one combined table.

In [16]:
main = pd.merge(orders,order_details,on= ["Order ID"],how="inner")

Shape of the merged table, just to confirm the merge did what it should.

In [17]:
main.shape

(1500, 10)

First few rows of the merged data.

In [18]:
main.head()

,Order ID,Order Date,CustomerName,State,City,Amount,Profit,Quantity,Category,Sub-Category
0,B-25601,2018-04-01,Bharat,Gujarat,Ahmedabad,1275.0,-1148.0,7,Furniture,Bookcases
1,B-25601,2018-04-01,Bharat,Gujarat,Ahmedabad,66.0,-12.0,5,Clothing,Stole
2,B-25601,2018-04-01,Bharat,Gujarat,Ahmedabad,8.0,-2.0,3,Clothing,Hankerchief
3,B-25601,2018-04-01,Bharat,Gujarat,Ahmedabad,80.0,-56.0,4,Electronics,Electronic Games
4,B-25602,2018-04-01,Pearl,Maharashtra,Pune,168.0,-111.0,2,Electronics,Phones


Sanity check — counting rows with negative Amount, negative Profit, or zero Quantity. Worth knowing these exist before analysis starts.

In [19]:
print("Rows with negative Amount:", (main["Amount"] < 0).sum())
print("Rows with negative Profit:", (main["Profit"] < 0).sum())
print("Rows with zero Quantity:", (main["Quantity"] == 0).sum())

Rows with negative Amount: 0
Rows with negative Profit: 503
Rows with zero Quantity: 0


Adding an Order Month column (year + month) so later grouping by month is easy.

In [20]:
main["Order Month"] = main["Order Date"].dt.to_period("M").dt.to_timestamp()

Info check for the sales_target table.

In [21]:
sales_target.info()

<class 'pandas.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   Month of Order Date  36 non-null     str    
 1   Category             36 non-null     str    
 2   Target               36 non-null     float64
dtypes: float64(1), str(2)
memory usage: 996.0 bytes


Sales target months are stored like 'Apr-18', so this parses them into proper dates.

In [22]:
sales_target["Month"] = pd.to_datetime(sales_target["Month of Order Date"],format= "%b-%y")

Same text cleanup for Category in the sales_target table.

In [23]:
sales_target["Category"] = sales_target["Category"].astype(str).str.strip().str.title()

Saving both cleaned tables to data/cleaned as CSVs.

In [ ]:
main.to_csv("data/cleaned/main.csv",index=False)
sales_target.to_csv("data/cleaned/sales_target.csv", index= False)

Setting up a connection to the local PostgreSQL database.

In [ ]:
from sqlalchemy import create_engine
engine = create_engine(
    "postgresql+psycopg2://e-commerce:(Your Password of pgadmin server and group)@localhost:5432/indian_e-commerce_analysis"
)

Pushing the cleaned main table into Postgres, replacing it if the table already exists.

In [ ]:
main.to_sql(
    "main",      # Table name in PostgreSQL
    engine,
    if_exists="replace", # replace / append / fail
    index=False
)

print("Data uploaded successfully!")

Data uploaded successfully!


Same thing for the sales_target table.

In [ ]:
sales_target.to_sql(
    "sales_target",      # Table name in PostgreSQL
    engine,
    if_exists="replace", # replace / append / fail
    index=False
)

print("Data uploaded successfully!")

Data uploaded successfully!


Final look at main after everything.

In [ ]:
main.head()

,Order ID,Order Date,CustomerName,State,City,Amount,Profit,Quantity,Category,Sub-Category,Order Month
0,B-25601,2018-04-01,Bharat,Gujarat,Ahmedabad,1275.0,-1148.0,7,Furniture,Bookcases,2018-04-01
1,B-25601,2018-04-01,Bharat,Gujarat,Ahmedabad,66.0,-12.0,5,Clothing,Stole,2018-04-01
2,B-25601,2018-04-01,Bharat,Gujarat,Ahmedabad,8.0,-2.0,3,Clothing,Hankerchief,2018-04-01
3,B-25601,2018-04-01,Bharat,Gujarat,Ahmedabad,80.0,-56.0,4,Electronics,Electronic Games,2018-04-01
4,B-25602,2018-04-01,Pearl,Maharashtra,Pune,168.0,-111.0,2,Electronics,Phones,2018-04-01


Final look at sales_target after everything.

In [ ]:
sales_target.head()

,Month of Order Date,Category,Target,Month
0,Apr-18,Furniture,10400.0,2018-04-01
1,May-18,Furniture,10500.0,2018-05-01
2,Jun-18,Furniture,10600.0,2018-06-01
3,Jul-18,Furniture,10800.0,2018-07-01
4,Aug-18,Furniture,10900.0,2018-08-01
